<a href="https://colab.research.google.com/github/visionbyangelic/African-Brain-Aging/blob/main/data/OASIS3_Filtering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# OASIS-3: Second Dataset Overview & Comparison with OpenBHB

## What is OASIS-3?

OASIS-3 (Open Access Series of Imaging Studies 3) is a large, publicly available neuroimaging dataset collected at the Washington University Knight Alzheimer Disease Research Center.

- **Total participants**: 1,378
- **Cognitively normal**: 755
- **Cognitive decline / Alzheimer’s spectrum**: 622
- **Age range**: 42–95 years
- **Design**: Longitudinal (participants scanned over many years)
- **Total MR sessions**: 2,842
- **Main MRI sequences**: T1-weighted, T2-weighted, FLAIR, ASL, SWI, resting-state BOLD, DTI
- **Other data**: FreeSurfer segmentations (available for many sessions), PET imaging (PiB, AV45, FDG)
- **Access**: Requires signing the OASIS Data Use Agreement and an approved account on NITRC-IR (nitrc.org/ir)
- **Download tools**: Official scripts available on the `oasis-scripts` GitHub repository

OASIS-3 is particularly valuable because it covers middle-aged and older adults, an age range that is under-represented in many open datasets.

---

## Why We Are Using It

Our project needs a clean set of **healthy-control** MRI scans to train a brain-age model.  
OASIS-3 supplies a large number of cognitively normal older adults. When combined with the younger OpenBHB sample, it will give us better coverage across the adult lifespan.

Only the **755 cognitively normal** participants will be retained for the normative baseline. Participants with cognitive impairment will be excluded.

---

## Key Differences from OpenBHB

| Feature                  | OpenBHB                                      | OASIS-3                                              |
|--------------------------|----------------------------------------------|------------------------------------------------------|
| **Focus**                | Healthy controls only                        | Mix of normal aging + cognitive decline              |
| **Sample size**          | ~3,984 public records (HC)                   | 1,378 total (755 cognitively normal)                 |
| **Age range**            | Mostly young (peak ~20 years, up to 88)      | Older adults (42–95 years)                           |
| **Design**               | Cross-sectional                              | Longitudinal (multiple visits per person)            |
| **Sites / scanners**     | Highly multi-site (10 source studies)        | Single center (Washington University)                |
| **Access method**        | Public on Hugging Face                       | NITRC-IR + Data Use Agreement required               |
| **Metadata availability**| Easy (participants.tsv + qc.tsv)             | Available as cohort/data spreadsheets on NITRC       |
| **Preprocessing**        | Uniform (CAT12 VBM, FreeSurfer, quasi-raw)   | FreeSurfer available for many sessions               |
| **Main challenge**       | Strong site/scanner effects                  | Handling longitudinal structure + clinical labels    |

---

## Planned Filtering Approach (Same Philosophy as OpenBHB)

We will follow a **metadata-first** strategy again:

1. Obtain the demographic and clinical spreadsheets (no heavy MRI files yet).
2. Keep only cognitively normal participants.
3. Decide how to treat longitudinal data (e.g., one session per person, or all sessions, or first visit only).
4. Inspect available scanner / field-strength information if present.
5. Produce a clean eligible-subject manifest before any image download.

This keeps the process efficient, auditable, and consistent with the OpenBHB pipeline we already completed.

---

## Current Status

- OpenBHB filtering is finished (Strict and Relaxed manifests ready).
- OASIS-3 has been identified and compared.
- Next step: begin metadata exploration once access and the relevant spreadsheet files are available.

In [24]:
# ============================================================
# OASIS-3 Filtering – Setup
# African Neurodata Research Lab | Brain Age Project
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np

# Same root used for OpenBHB
PROJECT_ROOT = Path('/content/drive/MyDrive/BrainAge_Data')
DATA_DIR     = PROJECT_ROOT / 'data' / 'oasis3'
MANIFESTS    = PROJECT_ROOT / 'manifests' / 'oasis3'

DATA_DIR.mkdir(parents=True, exist_ok=True)
MANIFESTS.mkdir(parents=True, exist_ok=True)

print("Project root :", PROJECT_ROOT)
print("Data folder  :", DATA_DIR)
print("Manifests    :", MANIFESTS)

Project root : /content/drive/MyDrive/BrainAge_Data
Data folder  : /content/drive/MyDrive/BrainAge_Data/data/oasis3
Manifests    : /content/drive/MyDrive/BrainAge_Data/manifests/oasis3


In [25]:
import zipfile
from pathlib import Path
import shutil

# Same root as OpenBHB
PROJECT_ROOT = Path('/content/drive/MyDrive/BrainAge_Data')
OASIS_DIR    = PROJECT_ROOT / 'data' / 'oasis3'
RAW_DIR      = OASIS_DIR / 'raw_download'      # original zip contents
MANIFESTS    = PROJECT_ROOT / 'manifests' / 'oasis3'

# Create clean folders
OASIS_DIR.mkdir(parents=True, exist_ok=True)
RAW_DIR.mkdir(parents=True, exist_ok=True)
MANIFESTS.mkdir(parents=True, exist_ok=True)

# Unzip into the controlled raw folder
zip_path = Path('/content/drive/MyDrive/BrainAge_Data/data/oasis3/raw_download/OASIS3_data_files')



In [26]:
from pathlib import Path
import shutil

RAW_DIR   = Path('/content/drive/MyDrive/BrainAge_Data/data/oasis3/raw_download')
CLEAN_DIR = Path('/content/drive/MyDrive/BrainAge_Data/data/oasis3')

# Map the long nested paths to clean filenames
file_map = {
    'OASIS3_demographics.csv':          'OASIS3_demographics.csv',
    'OASIS3_UDSa1_participant_demo.csv': 'OASIS3_UDSa1_demographics.csv',
    'OASIS3_UDSb4_cdr.csv':              'OASIS3_CDR.csv',
    'OASIS3_UDSd1_diagnoses.csv':        'OASIS3_diagnoses.csv',
    'OASIS3_Freesurfer_output.csv':      'OASIS3_Freesurfer.csv',
}

print("Copying to clean location...\n")
for src in RAW_DIR.rglob('*.csv'):
    name = src.name
    if name in file_map:
        dest = CLEAN_DIR / file_map[name]
        shutil.copy2(src, dest)
        print(f"  {file_map[name]}")

print("\nClean files now in:", CLEAN_DIR)

Copying to clean location...

  OASIS3_diagnoses.csv
  OASIS3_demographics.csv
  OASIS3_CDR.csv
  OASIS3_UDSa1_demographics.csv
  OASIS3_Freesurfer.csv

Clean files now in: /content/drive/MyDrive/BrainAge_Data/data/oasis3


In [27]:
import pandas as pd
from pathlib import Path

OASIS_DIR = Path('/content/drive/MyDrive/BrainAge_Data/data/oasis3')

# Load the key files
demo = pd.read_csv(OASIS_DIR / 'OASIS3_demographics.csv')
cdr  = pd.read_csv(OASIS_DIR / 'OASIS3_CDR.csv')
diag = pd.read_csv(OASIS_DIR / 'OASIS3_diagnoses.csv')

print("=== DEMOGRAPHICS ===")
print("Shape:", demo.shape)
print("Columns:", demo.columns.tolist())
print(demo.head(3))
print()

print("=== CDR (UDSb4) ===")
print("Shape:", cdr.shape)
print("Columns:", cdr.columns.tolist())
print(cdr.head(3))
print()

print("=== DIAGNOSES (UDSd1) ===")
print("Shape:", diag.shape)
print("Columns:", diag.columns.tolist())
print(diag.head(3))

=== DEMOGRAPHICS ===
Shape: (1378, 19)
Columns: ['OASISID', 'Subject_accession', 'AgeatEntry', 'AgeatDeath', 'GENDER', 'EDUC', 'SES', 'racecode', 'race', 'ETHNIC', 'AIAN', 'NHPI', 'ASIAN', 'AA', 'WHITE', 'daddem', 'momdem', 'HAND', 'APOE']
    OASISID  Subject_accession  AgeatEntry  AgeatDeath  GENDER  EDUC  SES  \
0  OAS30001                NaN     65.1945         NaN       2  12.0  4.0   
1  OAS30002                NaN     67.2521     76.9397       1  18.0  2.0   
2  OAS30003                NaN     58.8137         NaN       2  18.0  1.0   

   racecode   race  ETHNIC  AIAN  NHPI  ASIAN   AA  WHITE  daddem  momdem  \
0         5  White     0.0   0.0   0.0    0.0  0.0    1.0     5.0     1.0   
1         5  White     0.0   0.0   0.0    0.0  0.0    1.0     0.0     1.0   
2         5  White     0.0   0.0   0.0    0.0  0.0    1.0     0.0     0.0   

  HAND  APOE  
0    R  23.0  
1    R  34.0  
2    R  33.0  

=== CDR (UDSb4) ===
Shape: (8626, 23)
Columns: ['OASISID', 'OASIS_session_label',

Good data. Here’s what we now know:

### Summary of the three tables

| File | Rows | What it is | Key columns for us |
|------|------|------------|--------------------|
| **Demographics** | 1,378 | One row per person | `OASISID`, `AgeatEntry`, `GENDER` |
| **CDR** | 8,626 | Multiple visits per person | `OASISID`, `CDRTOT`, `dx1` |
| **Diagnoses** | 8,499 | Multiple visits per person | `OASISID`, `NORMCOG`, `DEMENTED` |

### How OASIS defines “cognitively normal”

The standard and cleanest rule used in OASIS papers is:

- **CDRTOT = 0** → cognitively normal  
  (Clinical Dementia Rating total score of zero)

There is also `NORMCOG = 1` in the diagnoses table, which should largely agree.


---
# OASIS-3 Healthy-Control Filter – Decision Record

## Project rule (from technical handoff)
- Unit of analysis = **participant**, not image/session
- No participant may appear more than once in the final normative set
- Training target = chronological age in cognitively normal individuals only

## OASIS-3 definition of cognitively normal
We use the standard OASIS criterion:

- **CDRTOT = 0** (Clinical Dementia Rating total score of zero)

This is the same definition used by the OASIS investigators to identify the 755 cognitively normal participants.

## Longitudinal handling
OASIS-3 contains multiple visits per person.  
To obey the participant-level rule:

1. Identify all visits where CDRTOT = 0
2. Keep **only one visit per unique OASISID**
3. Prefer the earliest visit with CDRTOT = 0 (can be updated later to the visit that has a usable T1)

## Final output of this stage
A clean manifest with:
- One row per cognitively normal participant
- Basic demographics (age, sex)
- CDR information for the retained visit
- Full audit trail of how the participant was selected

In [28]:
import pandas as pd
from pathlib import Path

OASIS_DIR  = Path('/content/drive/MyDrive/BrainAge_Data/data/oasis3')
MANIFESTS  = Path('/content/drive/MyDrive/BrainAge_Data/manifests/oasis3')
MANIFESTS.mkdir(parents=True, exist_ok=True)

# --------------------------------------------------
# 1. Load tables
# --------------------------------------------------
demo = pd.read_csv(OASIS_DIR / 'OASIS3_demographics.csv')
cdr  = pd.read_csv(OASIS_DIR / 'OASIS3_CDR.csv')

print("Demographics shape:", demo.shape)
print("CDR shape:         ", cdr.shape)

# --------------------------------------------------
# 2. Keep only cognitively normal visits (CDRTOT == 0)
# --------------------------------------------------
normal_visits = cdr[cdr['CDRTOT'] == 0].copy()
print(f"\nVisits with CDRTOT = 0: {len(normal_visits)}")
print(f"Unique participants with at least one CDRTOT = 0 visit: {normal_visits['OASISID'].nunique()}")

# --------------------------------------------------
# 3. One row per participant – keep earliest visit
# --------------------------------------------------
normal_visits = normal_visits.sort_values(['OASISID', 'days_to_visit'])
first_normal  = normal_visits.groupby('OASISID', as_index=False).first()

print(f"After keeping first CDRTOT = 0 visit per person: {len(first_normal)}")

# --------------------------------------------------
# 4. Merge basic demographics
# --------------------------------------------------
manifest = first_normal.merge(
    demo[['OASISID', 'AgeatEntry', 'GENDER', 'EDUC', 'HAND', 'APOE']],
    on='OASISID',
    how='left'
)

# --------------------------------------------------
# 5. Quick summary
# --------------------------------------------------
print("\n=== FINAL HEALTHY-CONTROL MANIFEST ===")
print(f"Participants retained: {len(manifest)}")
print(f"\nAge at entry:")
print(manifest['AgeatEntry'].describe())
print(f"\nSex (1=Male, 2=Female):")
print(manifest['GENDER'].value_counts(dropna=False))

# --------------------------------------------------
# 6. Save
# --------------------------------------------------
out_path = MANIFESTS / 'OASIS3_healthy_controls_manifest.csv'
manifest.to_csv(out_path, index=False)
print(f"\nSaved: {out_path}")

Demographics shape: (1378, 19)
CDR shape:          (8626, 23)

Visits with CDRTOT = 0: 6479
Unique participants with at least one CDRTOT = 0 visit: 1076
After keeping first CDRTOT = 0 visit per person: 1076

=== FINAL HEALTHY-CONTROL MANIFEST ===
Participants retained: 1076

Age at entry:
count    1076.000000
mean       67.654827
std         9.006537
min        42.498600
25%        62.832175
50%        68.437000
75%        73.418500
max        94.997300
Name: AgeatEntry, dtype: float64

Sex (1=Male, 2=Female):
GENDER
2    611
1    465
Name: count, dtype: int64

Saved: /content/drive/MyDrive/BrainAge_Data/manifests/oasis3/OASIS3_healthy_controls_manifest.csv


In [29]:
# --------------------------------------------------
# Consistency check with the diagnoses table
# --------------------------------------------------
diag = pd.read_csv(OASIS_DIR / 'OASIS3_diagnoses.csv')

# Keep only the same earliest visit we already selected
# (match on OASISID + days_to_visit)
check = manifest[['OASISID', 'days_to_visit', 'CDRTOT', 'dx1']].merge(
    diag[['OASISID', 'days_to_visit', 'NORMCOG', 'DEMENTED']],
    on=['OASISID', 'days_to_visit'],
    how='left'
)

print("Merged check shape:", check.shape)
print()

print("NORMCOG value counts (1 = cognitively normal):")
print(check['NORMCOG'].value_counts(dropna=False))
print()

print("DEMENTED value counts:")
print(check['DEMENTED'].value_counts(dropna=False))
print()

print("Cross-tab: CDRTOT vs NORMCOG")
print(pd.crosstab(check['CDRTOT'], check['NORMCOG'], dropna=False))
print()

print("Sample of rows where NORMCOG is not 1 (if any):")
print(check[check['NORMCOG'] != 1][['OASISID', 'days_to_visit', 'CDRTOT', 'dx1', 'NORMCOG', 'DEMENTED']].head(10))

Merged check shape: (1076, 6)

NORMCOG value counts (1 = cognitively normal):
NORMCOG
1.0    874
NaN    201
0.0      1
Name: count, dtype: int64

DEMENTED value counts:
DEMENTED
NaN    1068
0.0       6
1.0       2
Name: count, dtype: int64

Cross-tab: CDRTOT vs NORMCOG
NORMCOG  0.0  1.0  NaN
CDRTOT                
0.0        1  874  201

Sample of rows where NORMCOG is not 1 (if any):
     OASISID  days_to_visit  CDRTOT                 dx1  NORMCOG  DEMENTED
10  OAS30011              0     0.0  Cognitively normal      NaN       NaN
11  OAS30012              0     0.0  Cognitively normal      NaN       NaN
18  OAS30020              0     0.0  Cognitively normal      NaN       NaN
20  OAS30022              0     0.0  Cognitively normal      NaN       NaN
26  OAS30032              0     0.0  Cognitively normal      NaN       NaN
29  OAS30035            772     0.0  Cognitively normal      NaN       NaN
32  OAS30038              0     0.0  Cognitively normal      NaN       NaN
34  OAS30040

---
# OASIS-3 Healthy-Control Decision Note

## Final rule adopted for Phase 1

- Keep visits where **CDRTOT = 0**
- Retain **one visit per participant** (the earliest CDRTOT = 0 visit)
- Merge basic demographics

## Result

- **1,076 unique cognitively normal participants**
- Age range: 42.5 – 95.0 years (mean 67.7)
- Sex: 611 female / 465 male

## Consistency check

- 874 of the 1,076 also have NORMCOG = 1
- 201 have missing NORMCOG (not a contradiction)
- Only 1 case shows NORMCOG = 0 despite CDRTOT = 0

## Relation to the official “755” figure

The published OASIS-3 summary states 755 cognitively normal adults.  
Our count is higher because we used a clear, reproducible rule (any visit with CDRTOT = 0, first occurrence) rather than the stricter baseline/consensus definition behind the headline number.

For Phase 1 of the African Brain Age project this broader list is preferred because:

1. It maximises older-adult coverage (the main value OASIS-3 adds to OpenBHB)
2. It still obeys the project rule that the unit of analysis is the participant
3. The selection criterion is transparent and auditable
4. The list can be tightened later if a specific analysis requires closer alignment with the published 755

## File produced

`manifests/oasis3/OASIS3_healthy_controls_manifest.csv`

---
## What we did (simple version)

OASIS-3 has both healthy people and people with memory problems.

We kept only the healthy ones by using the clinical score called CDR.  
A CDR score of 0 means the person was judged cognitively normal.

Because many people were scanned more than once, we kept just **one visit per person** (the first time they had a CDR of 0).  
We do this so that each person counts only once — the same brain should not appear multiple times in the training data.

This gave us a clean list of **1,076 healthy older adults** (ages 42–95) that we can later use for the brain-age model.

---
# OASIS-3 Next Stage: Linking Healthy Controls to MRI Sessions

## Current status
- Clinical filter complete
- 1,076 unique cognitively normal participants retained (CDRTOT = 0, one visit each)
- Manifest saved: `OASIS3_healthy_controls_manifest.csv`

## Why this next step is needed
The 1,076 people were selected from clinical visits.  
In OASIS-3, the clinical assessment and the MRI scan are not always on the same day.

We now need to:
1. Find the actual MR sessions that belong to these 1,076 participants
2. Prefer sessions that are close in time to the clinical visit we kept
3. Confirm that a usable T1-weighted scan exists
4. Produce a final MRI-ready manifest

## Goal of this stage
Create a clean list of healthy-control participants who have both:
- Confirmed cognitive normality (CDRTOT = 0)
- At least one usable T1 MRI session

Only after this step will the OASIS-3 filtering be considered complete for Phase 1.

In [31]:
from pathlib import Path

OASIS_DIR = Path('/content/drive/MyDrive/BrainAge_Data/data/oasis3')

print("Files currently in data/oasis3:\n")
for f in sorted(OASIS_DIR.glob('*')):
    if f.is_file():
        print(f"  {f.name}  ({f.stat().st_size / 1024:.1f} KB)")

Files currently in data/oasis3:

  OASIS3_CDR.csv  (894.2 KB)
  OASIS3_Freesurfer.csv  (3489.6 KB)
  OASIS3_UDSa1_demographics.csv  (467.1 KB)
  OASIS3_demographics.csv  (76.2 KB)
  OASIS3_diagnoses.csv  (1843.3 KB)


In [32]:
from pathlib import Path

PROJECT_ROOT = Path('/content/drive/MyDrive/BrainAge_Data')
OASIS_DIR    = PROJECT_ROOT / 'data' / 'oasis3'
RAW_DIR      = OASIS_DIR / 'raw_download' / 'OASIS3_data_files'   # permanent location
MANIFESTS    = PROJECT_ROOT / 'manifests' / 'oasis3'

print("RAW_DIR exists:", RAW_DIR.exists())
print("\nContents of permanent raw folder:")
for f in sorted(RAW_DIR.rglob('*')):
    if f.is_file():
        print(" ", f.relative_to(RAW_DIR))

RAW_DIR exists: True

Contents of permanent raw folder:
  scans/FS-Freesurfer_output/resources/csv/files/OASIS3_Freesurfer_output.csv
  scans/UDSa1-Form_A1__Subject_Demographics/resources/csv/files/OASIS3_UDSa1_participant_demo.csv
  scans/UDSb4-Form_B4__Global_Staging__CDR__Standard_and_Supplemental/resources/csv/files/OASIS3_UDSb4_cdr.csv
  scans/UDSd1-Form_D1__Clinician_Diagnosis___Cognitive_Status_and_Dementia/resources/csv/files/OASIS3_UDSd1_diagnoses.csv
  scans/demo-demographics/resources/csv/files/OASIS3_demographics.csv


In [33]:
import pandas as pd
from pathlib import Path

OASIS_DIR = Path('/content/drive/MyDrive/BrainAge_Data/data/oasis3')
fs = pd.read_csv(OASIS_DIR / 'OASIS3_Freesurfer.csv')

print("FreeSurfer shape:", fs.shape)
print("\nColumns:")
print(fs.columns.tolist())
print("\nFirst 3 rows:")
print(fs.head(3))

FreeSurfer shape: (2681, 203)

Columns:
['Subject', 'MR_session', 'FS_FSDATA ID', 'Subject_accession', 'Freesurfer_accession', 'FS QC Status', 'version', 'IntraCranialVol', 'lhCortexVol', 'rhCortexVol', 'CortexVol', 'SubCortGrayVol', 'TotalGrayVol', 'SupraTentorialVol', 'lhCorticalWhiteMatterVol', 'rhCorticalWhiteMatterVol', 'CorticalWhiteMatterVol', '3rd-Ventricle_volume', '4th-Ventricle_volume', '5th-Ventricle_volume', 'Brain-Stem_volume', 'CC_Anterior_volume', 'CC_Central_volume', 'CC_Mid_Anterior_volume', 'CC_Mid_Posterior_volume', 'CC_Posterior_volume', 'CSF_volume', 'Left-Accumbens-area_volume', 'Left-Amygdala_volume', 'Left-Caudate_volume', 'Left-Cerebellum-White-Matter_volume', 'Left-Cerebellum-Cortex_volume', 'Left-choroid-plexus_volume', 'Left-Hippocampus_volume', 'Left-Inf-Lat-Vent_volume', 'Left-Lateral-Ventricle_volume', 'Left-non-WM-hypointensities_volume', 'Left-Pallidum_volume', 'Left-Putamen_volume', 'Left-Thalamus-Proper_volume', 'Left-VentralDC_volume', 'Left-vessel_

In [34]:
import pandas as pd
from pathlib import Path

OASIS_DIR = Path('/content/drive/MyDrive/BrainAge_Data/data/oasis3')
MANIFESTS = Path('/content/drive/MyDrive/BrainAge_Data/manifests/oasis3')

# Load previous healthy-control list and FreeSurfer table
hc = pd.read_csv(MANIFESTS / 'OASIS3_healthy_controls_manifest.csv')
fs = pd.read_csv(OASIS_DIR / 'OASIS3_Freesurfer.csv')

print("Healthy-control participants:", len(hc))
print("FreeSurfer rows (sessions):  ", len(fs))
print("Unique subjects in FreeSurfer:", fs['Subject'].nunique())

# Keep only FreeSurfer sessions that belong to our healthy controls
fs_hc = fs[fs['Subject'].isin(hc['OASISID'])].copy()
print(f"\nFreeSurfer sessions belonging to our 1,076 healthy controls: {len(fs_hc)}")
print(f"Unique healthy controls who have FreeSurfer: {fs_hc['Subject'].nunique()}")

# One session per person – keep the earliest MR session
# (MR_session ends with dXXXX = days from entry)
fs_hc['days_from_entry'] = fs_hc['MR_session'].str.extract(r'_d(\d+)').astype(float)
fs_hc = fs_hc.sort_values(['Subject', 'days_from_entry'])
fs_one = fs_hc.groupby('Subject', as_index=False).first()

print(f"After one session per person: {len(fs_one)}")

# Merge back the clinical info we already selected
final = fs_one.merge(
    hc[['OASISID', 'days_to_visit', 'age at visit', 'CDRTOT', 'AgeatEntry', 'GENDER']],
    left_on='Subject',
    right_on='OASISID',
    how='left'
)

print("\n=== FINAL MRI-LINKED HEALTHY CONTROLS ===")
print(f"Participants with both CDRTOT=0 and FreeSurfer/MRI: {len(final)}")
print(f"\nAge at entry:")
print(final['AgeatEntry'].describe())
print(f"\nSex (1=Male, 2=Female):")
print(final['GENDER'].value_counts(dropna=False))
print(f"\nFS QC Status:")
print(final['FS QC Status'].value_counts(dropna=False))

# Save
out = MANIFESTS / 'OASIS3_healthy_controls_with_MRI.csv'
final.to_csv(out, index=False)
print(f"\nSaved: {out}")

Healthy-control participants: 1076
FreeSurfer rows (sessions):   2681
Unique subjects in FreeSurfer: 1316

FreeSurfer sessions belonging to our 1,076 healthy controls: 2360
Unique healthy controls who have FreeSurfer: 1049
After one session per person: 1049

=== FINAL MRI-LINKED HEALTHY CONTROLS ===
Participants with both CDRTOT=0 and FreeSurfer/MRI: 1049

Age at entry:
count    1049.000000
mean       67.588327
std         8.999645
min        42.498600
25%        62.649300
50%        68.334200
75%        73.347900
max        94.997300
Name: AgeatEntry, dtype: float64

Sex (1=Male, 2=Female):
GENDER
2    601
1    448
Name: count, dtype: int64

FS QC Status:
FS QC Status
Passed               801
Passed with edits    239
Passed with Edits      9
Name: count, dtype: int64

Saved: /content/drive/MyDrive/BrainAge_Data/manifests/oasis3/OASIS3_healthy_controls_with_MRI.csv


/tmp/ipykernel_804/3423948883.py:24: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  fs_one = fs_hc.groupby('Subject', as_index=False).first()


# OASIS-3 Final Filter Note – MRI-Linked Healthy Controls

## Why this step was necessary

The previous list of 1,076 people came only from clinical visits (CDR scores).  
In OASIS-3 the clinical visit and the MRI scan are not always on the same day, and not every participant has usable imaging.

A brain-age model needs actual MRI data, not just a clinical label.  
Therefore we had to check which of the 1,076 cognitively normal participants also have a real structural MRI session.

## How we checked

We used the FreeSurfer table that was already downloaded.  
FreeSurfer is a processing pipeline that only runs on real MRI scans.  
If a participant appears in the FreeSurfer table, it means a structural MRI exists for them and has already been processed.

## Rule we applied

- Keep only participants who appear in the FreeSurfer table
- Keep **one MRI session per person** (the earliest one)
- This follows the project rule that the unit of analysis is the participant, not the session

## Result

| Stage | N |
|-------|---|
| Cognitively normal (CDRTOT = 0) | 1,076 |
| With FreeSurfer / MRI available | **1,049** |
| Lost (no FreeSurfer data) | 27 |

- Age range: 42.5 – 95.0 years (mean 67.6)
- Sex: 601 female / 448 male
- FreeSurfer QC: all sessions “Passed” or “Passed with edits” (none failed)

## File produced

`manifests/oasis3/OASIS3_healthy_controls_with_MRI.csv`

## Why this is the final Phase-1 list

We now have participants who satisfy both requirements of the project:

1. Confirmed cognitively normal (healthy controls)
2. Confirmed structural MRI available

